In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import pandas as pd
getfile = pd.read_csv('/content/drive/MyDrive/임실_데이터분석/임실군_관광지_리뷰데이터_1차전처리.csv')

In [ ]:
getfile.head(5)

# 리뷰 원문
  ↓
기본 정제
  ↓
Kiwi 형태소 분석
  ↓
불용어 제거
  ↓
토큰 생성
  ↓
scikit-learn
  ├─ Frequency
  ├─ TF-IDF
  └─ N-gram


```
# 코드로 형식 지정됨
```



# 기본 정제

In [ ]:
import pandas as pd
import re

# 원본 보존
df = getfile.copy()

def clean_text(text):
    if pd.isna(text):
        return None

    text = str(text)

    # 1. URL 제거
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 2. HTML 태그 제거
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. 줄바꿈 / 탭 → 공백
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # 4. 반복되는 특수문자 정리
    # 예: 좋아요!!!!! → 좋아요!
    text = re.sub(r'([!?~.,])\1+', r'\1', text)

    # 5. 키워드 분석에 불필요한 특수문자 제거
    # 한글 / 영어 / 숫자 / 기본 문장부호는 유지
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s!?.,]', ' ', text)

    # 6. 연속 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# body 원본은 그대로 두고 clean_body 추가
df['clean_body'] = df['body'].apply(clean_text)

In [ ]:
df[['body', 'clean_body']].head(20)
print("전체 리뷰:", len(df))
print("body 존재:", df['body'].notna().sum())
print("clean_body 존재:", df['clean_body'].notna().sum())
print("빈 문자열:", (df['clean_body'] == '').sum())

# kiwi 형태소 분석

In [ ]:
!pip install kiwipiepy

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

In [ ]:
TARGET_POS = {"NNG", "NNP", "VV", "VA"}

def kiwi_tokenize(text):
    if pd.isna(text):
        return []

    result = []

    for token in kiwi.tokenize(text):
        # VA-I, VV-R 같은 변형 태그까지 처리
        base_tag = token.tag.split("-")[0]

        if base_tag not in TARGET_POS:
            continue

        # 동사/형용사 → 기본형 형태
        if base_tag in {"VV", "VA"}:
            result.append(token.form + "다")
        else:
            result.append(token.form)

    return result


df["tokens"] = df["clean_body"].apply(kiwi_tokenize)

In [ ]:
df.loc[
    df["clean_body"].notna(),
    ["clean_body", "tokens"]
].sample(30, random_state=42)

## 1차 공통 불용어

In [ ]:
places = [
    "도화지 도예문화원",
    "세심자연휴양림",
    "임실치즈마을",
    "임실치즈테마파크",
    "전북119안전체험관",
    "전북종합사격장",
    "청소년수련원",
    "필봉농악전수교육관",
    "국립 임실호국원",
    "섬진강댐 물문화관",
    "소충사",
    "옥정호출렁다리",
    "전주샹그릴라CC",
    "옥정호",
    "사선대관광지",
    "임실역",
    "오수역",
    "임실시외버스터미널",
    "국사봉전망대",
    "관촌역",
    "붕어섬생태공원",
    "요산공원",
    "오수공용버스터미널",
    "관촌시외버스터미널",
    "임실전통시장",
    "사선대조각공원",
    "오수의견공원",
    "성수산왕의숲자연휴양림",
    "상이암",
    "옥정호작약꽃밭",
    "섬진강자전거길인증센터",
    "관촌시장",
    "강진공용버스터미널",
    "옥정호전망대"
]

In [ ]:
# 관광지명 사용자 사전 등록
for place in places:
    kiwi.add_user_word(place, "NNP")
    kiwi.add_user_word(place.replace(" ", ""), "NNP")

# 관광지명 사전 반영해서 tokens 다시 생성
df["tokens"] = df["clean_body"].apply(kiwi_tokenize)

In [ ]:
# 관광지명 자체 제거용
place_stopwords = set(places)
place_stopwords |= {p.replace(" ", "") for p in places}

# 1차 공통 불용어
common_stopwords = {
    "곳", "장소", "여기",
    "하다", "되다", "있다",
    "가다", "오다", "보다",
    "들다", "주다",
    "때", "것",
    "방문"
}

analysis_stopwords = {
    "좋다",
    "많다",
    "같다",
    "생각",
    "다녀오다",
    "나오다",
    "추천"
}


stopwords = common_stopwords | analysis_stopwords

In [ ]:
def remove_stopwords(tokens):
    return [
        token
        for token in tokens
        if token not in stopwords
        and len(token) > 1
    ]

df["keyword_tokens"] = df["tokens"].apply(remove_stopwords)

In [ ]:
from collections import Counter

all_tokens = [
    token
    for tokens in df["keyword_tokens"]
    for token in tokens
]

freq = Counter(all_tokens)
freq.most_common(50)

# 관광지별 frequency (A, B, C)

In [ ]:
# -----------------------------
# A / B / C 티어 관광지
# -----------------------------
A_places = [
    "붕어섬생태공원",
    "임실치즈테마파크",
    "옥정호출렁다리"
]

B_places = [
    "전주샹그릴라CC",
    "사선대관광지",
    "임실치즈마을",
    "전북119안전체험관",
    "임실전통시장"
]

C_places = [
    "섬진강댐 물문화관",
    "전북종합사격장",
    "국사봉전망대",
    "국립 임실호국원",
    "요산공원",
    "옥정호",
    "옥정호작약꽃밭",
    "상이암",
    "섬진강자전거길인증센터",
    "관촌시장",
    "필봉농악전수교육관",
    "오수의견공원",
    "성수산왕의숲자연휴양림"
]

ABC_places = A_places + B_places + C_places

In [ ]:
place_specific_stopwords = {
    # A tier
    "붕어섬생태공원": {"붕어섬", "임실"},
    "임실치즈테마파크": {"임실", "임실치즈", "테마파크"},
    "옥정호출렁다리": {"옥정호"},

    # B tier
    "전주샹그릴라CC": {"샹그릴라"},
    "사선대관광지": {"사선대", "임실"},
    "임실치즈마을": {"임실", "테마파크"},
    "전북119안전체험관": set(),
    "임실전통시장": {"임실"},

    # C tier
    "섬진강댐 물문화관": {"물문화관", "문화관"},
    "전북종합사격장": set(),
    "국사봉전망대": {"국사봉"},
    "국립 임실호국원": {"국립", "호국"},
    "요산공원": {"임실군"},
    "옥정호": set(),
    "옥정호작약꽃밭": set(),
    "상이암": {"상이암", "임실"},
    "섬진강자전거길인증센터": {"센터"},
    "관촌시장": set(),
    "필봉농악전수교육관": set(),
    "오수의견공원": set(),
    "성수산왕의숲자연휴양림": {"성수산", "자연휴양림", "휴양림"}
}

In [ ]:
def get_place_frequency(df, place_name, top_n=20):
    subset = df[df["place"] == place_name]

    extra_stopwords = place_specific_stopwords.get(place_name, set())

    tokens = [
        token
        for token_list in subset["keyword_tokens"]
        for token in token_list
        if token not in extra_stopwords
    ]

    return Counter(tokens).most_common(top_n)

In [ ]:
frequency_results = []

for place in ABC_places:
    result = get_place_frequency(df, place, 20)

    for rank, (keyword, count) in enumerate(result, start=1):
        frequency_results.append({
            "place": place,
            "tier": (
                "A" if place in A_places
                else "B" if place in B_places
                else "C"
            ),
            "rank": rank,
            "keyword": keyword,
            "count": count
        })

frequency_df = pd.DataFrame(frequency_results)

In [ ]:
# save_path = '/content/drive/MyDrive/임실_데이터분석/관광지별_frequency_top20.csv'

# frequency_df.to_csv(
#     save_path,
#     index=False,
#     encoding='utf-8-sig'
# )

# print(f"저장 완료: {save_path}")

# TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. 관광지별 추가 불용어까지 반영한 최종 토큰 생성
def apply_place_stopwords(row):
    tokens = row["keyword_tokens"]
    place = row["place"]

    extra_stopwords = place_specific_stopwords.get(place, set())

    return [
        token
        for token in tokens
        if token not in extra_stopwords
    ]

df["keyword_tokens_final"] = df.apply(
    apply_place_stopwords,
    axis=1
)

In [ ]:
# 2. A+B+C 전체 관광지를 TF-IDF corpus로 사용
tfidf_places = ABC_places

place_docs = (
    df[df["place"].isin(tfidf_places)]
    .groupby("place")["keyword_tokens_final"]
    .apply(
        lambda x: " ".join(
            token
            for token_list in x
            for token in token_list
        )
    )
)

In [ ]:
# 3. TF-IDF 계산
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(place_docs)

feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=place_docs.index,
    columns=feature_names
)

In [ ]:
# 4. 관광지별 TF-IDF Top 20
tfidf_results = []

for place in tfidf_places:

    if place not in tfidf_df.index:
        continue

    top20 = (
        tfidf_df.loc[place]
        .sort_values(ascending=False)
        .head(20)
    )

    tier = (
        "A" if place in A_places
        else "B" if place in B_places
        else "C"
    )

    for rank, (keyword, score) in enumerate(top20.items(), start=1):

        tfidf_results.append({
            "place": place,
            "tier": tier,
            "rank": rank,
            "keyword": keyword,
            "tfidf_score": score
        })

tfidf_result_df = pd.DataFrame(tfidf_results)

In [ ]:
# # 5. Google Drive 저장
# save_path = '/content/drive/MyDrive/임실_데이터분석/관광지별_tfidf_top20.csv'

# tfidf_result_df.to_csv(
#     save_path,
#     index=False,
#     encoding="utf-8-sig"
# )

# print(f"저장 완료: {save_path}")

# Frequency & TF-IDF 비교



In [ ]:
# 두 결과 병합
frequency_df = frequency_df.rename(columns={
    "count": "frequency"
})

compare_df = pd.merge(
    frequency_df,
    tfidf_result_df,
    on=["place", "tier", "keyword"],
    how="outer",
    suffixes=("_freq", "_tfidf")
)

compare_df = compare_df.rename(columns={
    "rank_freq": "freq_rank",
    "rank_tfidf": "tfidf_rank"
})

In [ ]:
# 관광지별 텍스트 리뷰 수

review_counts = (
    df[df["clean_body"].notna()]
    .groupby("place")
    .size()
    .rename("review_count")
)

compare_df = compare_df.merge(
    review_counts,
    on="place",
    how="left"
)

# 키워드 언급 비율
compare_df["freq_ratio"] = (
    compare_df["frequency"] / compare_df["review_count"]
)

In [ ]:
# # 저장
# save_path = "/content/drive/MyDrive/임실_데이터분석/관광지별_frequency_tfidf비교.csv"

# compare_df.to_csv(
#     save_path,
#     index=False,
#     encoding="utf-8-sig"
# )

In [ ]:
analysis_df = compare_df[
    compare_df["tier"].isin(["A", "B"])
].copy()

In [ ]:
# A/B 티어만
ab_freq = frequency_df[
    frequency_df["tier"].isin(["A", "B"])
].copy()

ab_tfidf = tfidf_result_df[
    tfidf_result_df["tier"].isin(["A", "B"])
].copy()

In [ ]:
for place in A_places + B_places:
    print(f"\n[{place} - Frequency Top 10]")
    display(
        ab_freq[ab_freq["place"] == place]
        .sort_values("rank")
        .head(10)[["rank", "keyword", "frequency"]]
    )

In [ ]:
for place in A_places + B_places:
    print(f"\n[{place} - TF-IDF Top 10]")
    display(
        ab_tfidf[ab_tfidf["place"] == place]
        .sort_values("rank")
        .head(10)[["rank", "keyword", "tfidf_score"]]
    )

In [ ]:
for place in A_places + B_places:
    print(f"\n[{place} - Frequency vs TF-IDF]")

    temp = compare_df[
        (compare_df["place"] == place) &
        (compare_df["tier"].isin(["A", "B"]))
    ].copy()

    display(
        temp[
            [
                "keyword",
                "frequency",
                "freq_rank",
                "tfidf_score",
                "tfidf_rank"
            ]
        ]
        .sort_values(
            ["freq_rank", "tfidf_rank"],
            na_position="last"
        )
        .head(20)
    )

-붕어섬생태공원: 다리, 붕어, 예쁘다, 산책이 핵심. 입장료, 생태, 볼거리는 Frequency보다 TF-IDF 순위가 상대적으로 높아서 특화성이 조금 더 있음.

-임실치즈테마파크: 치즈가 압도적 특화 키워드. 피자, 체험, 축제, 아이도 명확함.

-옥정호출렁다리: 다리, 붕어가 특화성이 매우 강함. 입장료, 건너다도 중요.

-전주샹그릴라CC: 그린, 캐디, 골프장, 코스, 페어웨이가 아주 깔끔하게 골프장 특성을 반영함.

-사선대관광지: 공원, 산책, 조각, 매운탕, 칼국수가 특징적.

-임실치즈마을: 치즈가 압도적이고, 피자, 체험, 맛있다가 뒤를 이음.

-전북119안전체험관: 체험이 매우 강하고, 안전, 아이, 유익, 예약, 재난 등이 잘 잡힘.

-임실전통시장: 시장, 마늘, 마늘쫑, 순대, 국밥 등이 잘 나와서 특색이 명확함.

한 가지 눈에 띄는 건 Frequency Top20과 TF-IDF Top20을 outer join 했기 때문에 NaN이 생기는 것인데, 이건 정상이고 문제 없어. 한쪽 Top20에는 있지만 다른 쪽 Top20에는 없다는 뜻.

# N-gram

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

## Bigram

In [ ]:
AB_places = A_places + B_places

bigram_results = []

for place in AB_places:

    subset = df[
        (df["place"] == place) &
        (df["keyword_tokens_final"].notna())
    ]

    counter = Counter()

    # 리뷰별로 bigram 생성
    for tokens in subset["keyword_tokens_final"]:

        if len(tokens) < 2:
            continue

        bigrams = [
            f"{tokens[i]} {tokens[i+1]}"
            for i in range(len(tokens) - 1)
        ]

        counter.update(bigrams)

    tier = "A" if place in A_places else "B"

    for rank, (bigram, count) in enumerate(
        counter.most_common(20),
        start=1
    ):
        bigram_results.append({
            "place": place,
            "tier": tier,
            "rank": rank,
            "bigram": bigram,
            "count": count
        })

bigram_df = pd.DataFrame(bigram_results)

In [ ]:
for place in AB_places:
    print(f"\n[{place} - Bigram Top 20]")

    display(
        bigram_df[
            bigram_df["place"] == place
        ]
    )

## Trigram

In [ ]:
trigram_results = []

for place in A_places:

    subset = df[
        (df["place"] == place) &
        (df["keyword_tokens_final"].notna())
    ]

    counter = Counter()

    # 리뷰별로 trigram 생성
    for tokens in subset["keyword_tokens_final"]:

        if len(tokens) < 3:
            continue

        trigrams = [
            f"{tokens[i]} {tokens[i+1]} {tokens[i+2]}"
            for i in range(len(tokens) - 2)
        ]

        counter.update(trigrams)

    for rank, (trigram, count) in enumerate(
        counter.most_common(20),
        start=1
    ):
        trigram_results.append({
            "place": place,
            "tier": "A",
            "rank": rank,
            "trigram": trigram,
            "count": count
        })

trigram_df = pd.DataFrame(trigram_results)

In [ ]:
# bigram_df.to_csv(
#     "/content/drive/MyDrive/임실_데이터분석/관광지별_bigram_top20.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# trigram_df.to_csv(
#     "/content/drive/MyDrive/임실_데이터분석/관광지별_trigram_top20.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

In [ ]:
# save_path = "/content/drive/MyDrive/임실_데이터분석/리뷰데이터_키워드전처리완료.csv"

# df.to_csv(
#     save_path,
#     index=False,
#     encoding="utf-8-sig"
# )